In [5]:
import torch
import torch.nn as nn

In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.softmax = nn.Softmax(dim=-1)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, X):
        Q = self.q_proj(X)
        V = self.v_proj(X)
        K = self.k_proj(X)

        dim_model_single_head = self.d_model // self.num_heads

        list_Q = []
        list_V = []
        list_K = []

        for i in range(0, Q.shape[1], dim_model_single_head):
            list_Q.append(Q[:, i:i+dim_model_single_head])
            list_K.append(K[:, i:i+dim_model_single_head])
            list_V.append(V[:, i:i+dim_model_single_head])

        slices_with_attention = []

        for i in range(len(list_Q)):
            initial = torch.matmul(list_Q[i], list_K[i].T)
            
            normalized = self.softmax(torch.div(initial, torch.sqrt(torch.tensor(dim_model_single_head, dtype=torch.float32))))

            slices_with_attention.append(torch.matmul(normalized, list_V[i]))

        concatenated = torch.concat(slices_with_attention, dim=1)

        return self.out_proj(concatenated)

In [14]:
torch.manual_seed(42)
mha = MultiHeadAttention(d_model=4, num_heads=2)

X = torch.tensor([[1, 0, 1, 0],
                [0, 1, 0, 1],
                [1, 1, 0, 0]], dtype=torch.float32)

output = mha(X)
print(output)

tensor([[ 0.4267, -0.2835, -0.1070,  0.0056],
        [ 0.4523, -0.3099, -0.0707,  0.0213],
        [ 0.4374, -0.2942, -0.0927,  0.0108]], grad_fn=<MmBackward0>)
